# **Penting**
- Jangan mengubah atau menambahkan cell text yang sudah disediakan, Anda hanya perlu mengerjakan cell code yang sudah disediakan.
- Pastikan seluruh kriteria memiliki output yang sesuai, karena jika tidak ada output dianggap tidak selesai.
- Misal, Anda menggunakan df = df.dropna() silakan gunakan df.isnull().sum() sebagai tanda sudah berhasil. Silakan sesuaikan seluruh output dengan perintah yang sudah disediakan.
- Pastikan Anda melakukan Run All sebelum mengirimkan submission untuk memastikan seluruh cell berjalan dengan baik.
- Pastikan Anda menggunakan variabel df dari awal sampai akhir dan tidak diperbolehkan mengganti nama variabel tersebut.
- Hapus simbol pagar (#) pada kode yang bertipe komentar jika Anda menerapkan kriteria tambahan
- Biarkan simbol pagar (#) jika Anda tidak menerapkan kriteria tambahan
- Pastikan Anda mengerjakan sesuai section yang sudah diberikan tanpa mengubah judul atau header yang disediakan.

# **1. Import Library**
Pada tahap ini, Anda perlu mengimpor beberapa pustaka (library) Python yang dibutuhkan untuk analisis data dan pembangunan model machine learning.

In [1]:
# Library essential
import numpy as np
import pandas as pd

# Library pre-processing data
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

# Library ML
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

# Library evaluasi
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Library save model (joblib)
import joblib

# Library hyperparameter tuning
!pip install scikit-optimize
from skopt import BayesSearchCV

# **2. Memuat Dataset dari Hasil Clustering**
Memuat dataset hasil clustering dari file CSV ke dalam variabel DataFrame.

In [2]:
# Gunakan dataset hasil clustering yang memiliki fitur Target
# Silakan gunakan dataset data_clustering jika tidak menerapkan Interpretasi Hasil Clustering [Advanced]
# Silakan gunakan dataset data_clustering_inverse jika menerapkan Interpretasi Hasil Clustering [Advanced]
# Lengkapi kode berikut
df = pd.read_csv("data_clustering_inverse.csv")

In [3]:
# Tampilkan 5 baris pertama dengan function head.
df.head()

,TransactionAmount,TransactionDate,TransactionType,Location,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,PreviousTransactionDate,bin_age,Target
0,14.09,2023-04-11 16:29:14,Debit,San Diego,ATM,70.000000,Doctor,81.0,1.0,5112.21,2024-11-04 08:08:08,"(0.8, 1.0]",1
1,376.24,2023-06-27 16:44:19,Debit,Houston,ATM,68.000000,Doctor,141.0,1.0,13758.91,2024-11-04 08:09:35,"(0.8, 1.0]",0
2,126.29,2023-07-10 18:16:08,Debit,Mesa,Online,19.000000,Student,56.0,1.0,1122.35,2024-11-04 08:07:04,"(-0.001, 0.2]",0
3,184.50,2023-05-05 16:32:11,Debit,Raleigh,Online,26.000000,Student,25.0,1.0,8569.06,2024-11-04 08:09:06,"(-0.001, 0.2]",1
4,13.45,2023-10-16 17:51:24,Credit,Atlanta,Online,44.678444,Student,198.0,1.0,7429.40,2024-11-04 08:06:39,"(0.4, 0.6]",2


# **3. Data Splitting**
Tahap Data Splitting bertujuan untuk memisahkan dataset menjadi dua bagian: data latih (training set) dan data uji (test set).

In [4]:
# Menggunakan train_test_split() untuk melakukan pembagian dataset.

# Kolom Target dipisahkan, dan kolom tanggal di-drop agar tidak mengganggu hasil klasifikasi
X = df.drop(columns=['Target', 'TransactionDate', 'PreviousTransactionDate'])
y = df['Target']

# Sebelum splitting, data di-encoding dan di-scaling agar bisa diproses; sama seperti Clustering sebelumnya
# Encoding & scaling dilakukan pada X (training set) agar target tidak di-scaling oleh MinMaxScaler
kategorikal = X.select_dtypes(include=['object']).columns
numerik = X.select_dtypes(include=['int64', 'float64']).columns

encoders = {}
for col in kategorikal:
  label_encoder = LabelEncoder()
  X[col] = label_encoder.fit_transform(X[col])
  encoders[col] = label_encoder

scaler = MinMaxScaler()
X[numerik] = scaler.fit_transform(X[numerik])


# Split data menjadi set pelatihan dan set uji
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training dataset: X_train =", X_train.shape)
print("Training dataset: y_train =", y_train.shape)
print("Test set shape: X_test =", X_test.shape)
print("Test set shape: y_test =", y_test.shape)

X.describe()

Training dataset: X_train = (1813, 10)
Training dataset: y_train = (1813,)
Test set shape: X_test = (454, 10)
Test set shape: y_test = (454,)


,TransactionAmount,TransactionType,Location,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,bin_age
count,2267.000000,2267.000000,2267.000000,2267.000000,2267.000000,2267.000000,2267.000000,2267.0,2267.000000,2267.000000
mean,0.284894,0.788708,21.344067,1.011910,0.429590,1.531098,0.375335,0.0,0.337183,1.655492
std,0.241027,0.439545,12.432512,0.827772,0.284355,1.147651,0.240138,0.0,0.259119,1.399776
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000
25%,0.087337,1.000000,11.000000,0.000000,0.145161,1.000000,0.182759,0.0,0.095788,0.000000
50%,0.224276,1.000000,21.000000,1.000000,0.435484,2.000000,0.351724,0.0,0.319879,2.000000
75%,0.415151,1.000000,32.000000,2.000000,0.661290,3.000000,0.520690,0.0,0.504446,3.000000
max,1.000000,2.000000,43.000000,3.000000,1.000000,4.000000,1.000000,0.0,1.000000,4.000000


# **4. Membangun Model Klasifikasi**
Setelah memilih algoritma klasifikasi yang sesuai, langkah selanjutnya adalah melatih model menggunakan data latih.

Berikut adalah rekomendasi tahapannya.
1. Menggunakan algoritma klasifikasi yaitu Decision Tree.
2. Latih model menggunakan data yang sudah dipisah.

In [5]:
# Buatlah model klasifikasi menggunakan Decision Tree
d_tree = DecisionTreeClassifier()
d_tree.fit(X_train, y_train)

DecisionTreeClassifier()

In [6]:
# Menyimpan Model
# import joblib
joblib.dump(d_tree, 'decision_tree_model.h5')

['decision_tree_model.h5']

# **5. Memenuhi Kriteria Skilled dan Advanced dalam Membangun Model Klasifikasi**



**Biarkan kosong jika tidak menerapkan kriteria skilled atau advanced**

In [7]:
# Melatih model menggunakan algoritma klasifikasi scikit-learn selain Decision Tree.
knn = KNeighborsClassifier()
knn.fit(X_train, y_train)
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

RandomForestClassifier()

In [8]:
# Menampilkan hasil evaluasi akurasi, presisi, recall, dan F1-Score pada seluruh algoritma yang sudah dibuat.
prediksi_d_tree = d_tree.predict(X_test)

akurasi_d_tree = accuracy_score(y_test, prediksi_d_tree)
presisi_d_tree = precision_score(y_test, prediksi_d_tree, average='weighted')
recall_d_tree = recall_score(y_test, prediksi_d_tree, average='weighted')
f1_score_d_tree = f1_score(y_test, prediksi_d_tree, average='weighted')


prediksi_knn = knn.predict(X_test)

akurasi_knn = accuracy_score(y_test, prediksi_knn)
presisi_knn = precision_score(y_test, prediksi_knn, average='weighted')
recall_knn = recall_score(y_test, prediksi_knn, average='weighted')
f1_score_knn = f1_score(y_test, prediksi_knn, average='weighted')

prediksi_rf = rf.predict(X_test)

akurasi_rf = accuracy_score(y_test, prediksi_rf)
presisi_rf = precision_score(y_test, prediksi_rf, average='weighted')
recall_fr = recall_score(y_test, prediksi_rf, average='weighted')
f1_score_rf = f1_score(y_test, prediksi_rf, average='weighted')

hasil = {
    "Akurasi": [akurasi_d_tree, akurasi_knn, akurasi_rf],
    "Presisi": [presisi_d_tree, presisi_knn, presisi_rf],
    "Recall": [recall_d_tree, recall_knn, recall_fr],
    "F1 Score": [f1_score_d_tree, f1_score_knn, f1_score_rf]
}

df_hasil = pd.DataFrame(hasil, index=['Decision Tree', 'K-Nearest Neighbor', 'Random Forest'])
df_hasil

,Akurasi,Presisi,Recall,F1 Score
Decision Tree,0.277533,0.277289,0.277533,0.277152
K-Nearest Neighbor,0.310573,0.310063,0.310573,0.307435
Random Forest,0.339207,0.339354,0.339207,0.338598


In [9]:
# Menyimpan Model Selain Decision Tree
# Model ini bisa lebih dari satu
# import joblib
joblib.dump(knn, 'explore_knn_classification.h5')
joblib.dump(rf, 'explore_random_forest_classification.h5')

['explore_random_forest_classification.h5']

Hyperparameter Tuning Model

Pilih salah satu algoritma yang ingin Anda tuning

In [10]:
# Lakukan Hyperparameter Tuning dan Latih ulang.
# Lakukan dalam satu cell ini saja.

# Hyperparameter tuning hanya dilakukan pada 1 algoritma (sesuai kriteria), yaitu Decision Tree
param_space  = {
    'criterion': ['gini', 'entropy'],
    'max_depth': (25, 100),
    'min_samples_split': (2,10),
    'min_samples_leaf': (1, 10)
}

bayes_search = BayesSearchCV(estimator=d_tree, search_spaces=param_space,random_state=42)
bayes_search.fit(X_train, y_train)
print("Parameter terbaik:",  bayes_search.best_params_)

crit = bayes_search.best_params_['criterion']
max_d = bayes_search.best_params_['max_depth']
min_s_s = bayes_search.best_params_['min_samples_split']
min_s_l = bayes_search.best_params_['min_samples_leaf']

d_tree_hyper = DecisionTreeClassifier(criterion=crit, max_depth=max_d, min_samples_leaf=min_s_s, min_samples_split=min_s_l)
d_tree_hyper.fit(X_train, y_train)

Parameter terbaik: OrderedDict({'criterion': 'gini', 'max_depth': 70, 'min_samples_leaf': 3, 'min_samples_split': 2})


DecisionTreeClassifier(max_depth=70, min_samples_leaf=2, min_samples_split=3)

In [11]:
# Menampilkan hasil evaluasi akurasi, presisi, recall, dan F1-Score pada algoritma yang sudah dituning.

prediksi_d_tree_hyper = d_tree_hyper.predict(X_test)

akurasi_d_tree_hyper = accuracy_score(y_test, prediksi_d_tree_hyper)
presisi_d_tree_hyper = precision_score(y_test, prediksi_d_tree_hyper, average='weighted')
recall_d_tree_hyper = recall_score(y_test, prediksi_d_tree_hyper, average='weighted')
f1_score_d_tree_hyper = f1_score(y_test, prediksi_d_tree_hyper, average='weighted')

hasil_hyper = {
    "Akurasi": [akurasi_d_tree_hyper],
    "Presisi": [presisi_d_tree_hyper],
    "Recall": [recall_d_tree_hyper],
    "F1 Score": [f1_score_d_tree_hyper]
}
df_hyper = pd.DataFrame(hasil_hyper)
df_hyper

# Note: Setelah tuning hyperparameter, skor masih sangat rendah pada sekitar 0,3x.
# Sepertinya model mengalami overfitting, sehingga seluruh nilai evaluasinya rendah.


,Akurasi,Presisi,Recall,F1 Score
0,0.303965,0.299387,0.303965,0.3001


In [12]:
# Menyimpan Model hasil tuning
# import joblib
joblib.dump(d_tree_hyper, 'tuning_classification.h5')

['tuning_classification.h5']

End of Code